## Demo for user story 569: staging of a item between s3 buckets using credentials

Link to the story: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-569   
This notebook shows an example of a staging directly from an external s3 bucket.

### Setup environment and Dask cluster

In [ ]:
import requests
import os
import pprint
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client, *_ = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10

In [ ]:
# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

In [ ]:
# Create a test collection 
collection = create_test_collection()

# Check the catalog for my_test_collection
collection = catalog_client.get_collection(TEST_COLLECTION)
for item in collection.get_items():
    print(f"Item {item.id} has {len(item.assets)} assets")

### Check jobs and staging
Check that the staging process is ready and that the jobs table is empty

In [ ]:
# Existing process
process = staging_client.get_process("staging")
print(f"Staging process information: {process}")

jobs = staging_client.get_jobs()
if jobs.get("numberMatched") > 0:
    delete_jobs = True
    if cluster_mode == True:
        delete_jobs = input(f"There are {jobs.get('numberMatched')} jobs in the table. Do you want to delete them all (y/n)?").lower().strip() == 'y'
    if delete_jobs:
        print("Deleting all the jobs...")
        for job in jobs.get("jobs"):
            delete_response = staging_client.delete_job(job.get('jobID'))
        # Check that the jobs have been deleted
        jobs = staging_client.get_jobs()
        print(f"Existing jobs: {jobs}")

### Create an incomplete STAC description for the item to stage

For this example we are going to stage one item from one of our s3 buckets.   
The item is s3://rs-dev-cluster-temp/stations/CADIP/S1A_20240410083700053369.short/ch_1/DCS_01_S1A_20240410083700053369_ch1_DSDB_00001.raw   
We manually create a STAC description for this item, fitting with the specifications from the story.   
To test an error case, this first description lacks the correct storage scheme for the external s3.   
The Feature representing the item contains two storage schemes: one that exists (test-s3), and one that doesn't exist, to show the adaptability in the solution.

In [ ]:
demo_feature_stac = {
  "id": "S1A_EW_GRDM_1SDH_20250519T121832_20250519T121932_059263_075AAE_F3E5_COG",
  "type": "Feature",
  "geometry": None,
  "assets": {
    "tst1": {
      "href": "s3://rs-dev-cluster-temp/stations/CADIP/S1A_20240410083700053369.short/ch_1/DCS_01_S1A_20240410083700053369_ch1_DSIB.xml",
      "file:size": 538,
      "file:local_path": "S1A_20240410083700053369.short/ch_1/DCS_01_S1A_20240410083700053369_ch1_DSIB.xml",
      "auth:refs": [
        "s3"
      ],
      "storage:refs": [
        "unknown-s3"
      ]
    }
  },
  "collection": "sentinel-1-grd",
  "properties": {
    "datetime": "2025-06-25T11:21:26.165Z",
    "auth:schemes": {
      "s3": {
        "type": "s3"
      }
    },
    "storage:schemes": {
      "unknown-s3": {
        "type": "custom-s3",
        "title": "S3 that doesn't exist",
        "platform": "https://some.url.com",
        "description": "Endpoint that doesn't exist, for test purpose.",
        "requester_pays": True
      }
    }
  },
  "links": [
    {
      "rel": "root",
      "type": "application/json",
      "href": "https://dev-rspy-ovh.esa-copernicus.eu/cadip/"
    },
    {
      "rel": "self",
      "type": "application/geo+json",
      "href": "https://dev-rspy-ovh.esa-copernicus.eu/cadip/collections/sgs/items/S1A_20240410083700053369"
    },
    {
      "rel": "collection",
      "type": "application/json",
      "href": "https://dev-rspy-ovh.esa-copernicus.eu/cadip/collections/sgs"
    },
  ],
  "stac_extensions": [
    "https://stac-extensions.github.io/authentication/v1.1.0/schema.json",
    "https://stac-extensions.github.io/file/v2.1.0/schema.json",
    "https://stac-extensions.github.io/storage/v2.0.0/schema.json",
  ],
  "stac_version": "1.1.0"
}

### Test error case

Here we run a job with the previous STAC specification. This job is expected to fail as the specification lacks the correct storage scheme.

In [ ]:
# Run one staging for the test item
staging_job = staging_client.run_staging(demo_feature_stac, TEST_COLLECTION)

In [ ]:
try:
    staging_client.wait_for_jobs(staging_job, logger)
    assert False
except RuntimeError:
    print("Job failed as expected")
    assert True

In [ ]:
# Check the job previously launched is failed and why
job_id = staging_job['rs-dev-cluster-temp']['jobID']
job_results = staging_client.get_job_info(job_id)
print(f"Results from job {job_id}: {job_results['status']}")
print(f"Reason of fail: {job_results['message']}")
assert job_results['status'] == "failed"

### Update STAC input

Update STAC item with a storage scheme corresponding to our credentials for the s3 endpoint.

In [ ]:
test_s3_storage_scheme = {
    "type": "custom-s3",
    "title": "Our Test S3 bucket",
    "platform": "https://s3.gra.io.cloud.ovh.net",
    "description": "Endpoint to our internal test s3 bucket.",
    "requester_pays": False
}
demo_feature_stac["assets"]["tst1"]["storage:refs"].append("test-s3")
demo_feature_stac["properties"]["storage:schemes"]["test-s3"] = test_s3_storage_scheme

### Create and run a successful job
Create a staging job, run it, and make sure it is successful.

In [ ]:
# Run one staging for the test item
staging_job = staging_client.run_staging(demo_feature_stac, TEST_COLLECTION)

In [ ]:
staging_client.wait_for_jobs(staging_job, logger)

In [ ]:
# Check the job previously launched is successful
job_id = staging_job['rs-dev-cluster-temp']['jobID']
job_results = staging_client.get_job_results(job_id)
print(f"Results from job {job_id}: {job_results}")
assert job_results == "successful"

### Check that the catalog contains one item

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_collection(TEST_COLLECTION).get_items())
print (f"{len(result)} items before removing")
print(f"Item {result[0].id} has {len(result[0].assets)} assets")
assert len(result) == 1

### End of demo
Delete the whole collection and shutdown the Dask cluster

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())

In [ ]:
shutdown = False
if shutdown:
    # Shutdown dask cluster staging
    shutdown_dask_cluster_staging()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.